# 02 Cost-Latency Policy Routing (LiteLLM, 2026)

## What This Lesson Is
Build a policy function that selects a model from cost and latency budgets.

## Scientific Lens
- Concept: Multi-objective decision policy for model routing
- Measure: Composite score and policy compliance rate
- Validity Limit: Synthetic pricing and latency tables must be replaced with real telemetry for production decisions.


## How It Works
1. Create weighted scoring for cost and latency.
2. Apply hard constraints (budget/SLA).
3. Measure real latency and token usage from live calls.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Latency SLA (ms):", 1200)
print("Cost budget ($/1k tokens):", 0.5)


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
candidates = [
    {"model": "gpt-4.1-mini", "cost_per_1k": 0.40, "p95_ms": 900},
    {"model": "gpt-4o-mini", "cost_per_1k": 0.25, "p95_ms": 700},
    {"model": "local-small", "cost_per_1k": 0.02, "p95_ms": 1600},
]

sla_ms = 1200
budget = 0.5
weight_cost, weight_latency = 0.6, 0.4

feasible = [c for c in candidates if c["p95_ms"] <= sla_ms and c["cost_per_1k"] <= budget]
for c in feasible:
    c["score"] = (weight_cost * c["cost_per_1k"]) + (weight_latency * (c["p95_ms"] / 1000))

selected = sorted(feasible, key=lambda x: x["score"])[0]
print(selected)
assert selected["model"] == "gpt-4o-mini"


In [ ]:
# Live Demo
import os
import time

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live measurement: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live measurement: OPENAI_API_KEY not set.")
    else:
        for model in ["openai/gpt-4.1-mini", "openai/gpt-4o-mini"]:
            t0 = time.perf_counter()
            r = completion(
                model=model,
                messages=[{"role": "user", "content": "Explain latency-vs-cost routing in 12 words."}],
                api_key=api_key,
                timeout=20,
            )
            dt_ms = round((time.perf_counter() - t0) * 1000, 1)
            usage = getattr(r, "usage", None)
            print({"model": model, "latency_ms": dt_ms, "usage": str(usage)})


## Applied Labs
1. Change policy weights from 60/40 to 30/70 and explain model selection changes.
2. Add a hard quality threshold and reject candidates below that threshold.
3. Capture 5 live latency samples per model and compare median vs p95 behavior.

## Validation Checklist
- Policy enforces both hard constraints and weighted optimization.
- Selected model is explainable from cost/latency math.
- Live metrics include latency and usage visibility.

## Further Reading
- [LiteLLM Router Guide](https://docs.litellm.ai/docs/routing)
- [OpenAI API Pricing Concepts](https://platform.openai.com/docs/pricing)
- [SRE Latency Percentiles](https://sre.google/sre-book/monitoring-distributed-systems/)
